# WithU Talk — Phase 1: Two-Stage Text Cyberbullying Detector

Trains the **verbal-violence text head** — the workhorse module — reading the exact files
produced by the Phase 0–1 prep notebook (`./prepared/`).

**Two-stage transfer learning:**
1. **Stage-1 (general toxicity):** fine-tune KcELECTRA on the public pool (BEEP + K-MHaS) →
   learns broad Korean toxic/clean from ~94K balanced examples.
2. **Stage-2 (domain adaptation):** continue training on the WithU domain data →
   shifts the model from news-comment register to elementary group-chat language.

Output: a saved model that maps a message → **P(toxic)**, plus an honest evaluation on your
event-level domain test set and on the (statistically-powered) public test set.

> **Runtime note.** The training cells need **PyTorch + a GPU** and will **download the base
> model from Hugging Face on first run** (needs internet). On CPU, use `SMOKE_TEST = True` first.
> The data-loading, class-weight, and metric logic here were verified against your real splits;
> the GPU training loop follows the standard 🤗 `Trainer` pattern.


In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"     # expose only ONE GPU to this kernel
for k in ("RANK", "LOCAL_RANK", "WORLD_SIZE", "MASTER_ADDR", "MASTER_PORT"):
    os.environ.pop(k, None)

In [10]:
# Install (run once). Torch: match your CUDA — see https://pytorch.org for the right command.
# pip install torch --index-url https://download.pytorch.org/whl/cu121   # example for CUDA 12.1
# pip install transformers datasets scikit-learn pandas pyarrow accelerate
import os, json, numpy as np, pandas as pd
from pathlib import Path
from sklearn.metrics import (precision_recall_fscore_support, accuracy_score,
                             confusion_matrix, classification_report, precision_recall_curve)

import torch, torch.nn as nn
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from datasets import Dataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| GPUs:", torch.cuda.device_count() if DEVICE=="cuda" else 0)
if DEVICE == "cpu":
    print("⚠️  No GPU detected — full training will be slow. Set SMOKE_TEST=True to validate the pipeline first.")

Device: cuda | GPUs: 1


In [3]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "—")
!nvidia-smi

True NVIDIA H200 MIG 4g.71gb
Wed Jul 15 10:30:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H200                    On  |   00000000:DB:00.0 Off |                   On |
| N/A   27C    P0             86W /  700W |     105MiB / 143771MiB |     N/A      Default |
|                                         |                        |              Enabled |
+------------------

In [7]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

In [4]:
# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------
PREP_DIR   = Path("./prepared")                 # from Phase 0-1 Downloads/prepared/stage1_public.parquet
MODEL_NAME = "beomi/KcELECTRA-base"             # informal-Korean specialist; swap to "monologg/kobert" if desired
OUT_DIR    = Path("./models")
STAGE1_DIR = OUT_DIR / "stage1_general"
STAGE2_DIR = OUT_DIR / "stage2_domain_final"

MAX_LEN    = 128        # chat messages are short
SMOKE_TEST = False      # True -> tiny subset + 1 epoch, to verify the whole run end-to-end fast

# Stage-1 (public) hyperparameters
S1_EPOCHS, S1_BS, S1_LR = 3, 32, 2e-5
# Stage-2 (domain) — lower LR, fewer epochs to adapt without forgetting
S2_EPOCHS, S2_LR = 2, 1e-5
S2_NEG_PER_POS = 8      # subsample domain negatives to this ratio so 467 positives aren't drowned

SEED = 42
OUT_DIR.mkdir(parents=True, exist_ok=True)
torch.manual_seed(SEED); np.random.seed(SEED)

In [5]:
import os
print(os.getcwd())              # where the kernel actually is
print(os.path.exists("your/path/here"))   # True/False, no guessing
print(os.listdir("."))          # what files the kernel can actually see

/home/yonsei/Downloads
False
['.~lock.excel_wangtta.xlsx#', '020.주제별 텍스트 일상 대화 데이터-20260709T032547Z-3-001.zip', 'output_folder', 'excel_wangtta.xlsx', 'aihub', 'INNORIX-EX-Agent-22.deb', 'data', 'withu_phase0_1_data_prep.ipynb', '.ipynb_checkpoints', 'withu_phase1_text_detector.ipynb', 'models', 'input_folder', '012.한국어 SNS 멀티턴 대화 데이터-20260709T032740Z-3-001.zip', 'prepared', 'Downloads']


## 1. Load the prepared splits → 🤗 Datasets

Handles the two data quirks we saw: domain `cb_binary` is float with a couple of null texts, so we
drop nulls and cast to int. Each split becomes a `Dataset` with `text` + `labels`.


In [6]:
def load_split(parquet_path, split):
    df = pd.read_parquet(parquet_path)
    df = df[df.split == split][["text", "cb_binary"]].copy()
    df = df.dropna(subset=["text", "cb_binary"])
    df = df[df.text.str.strip().astype(bool)]           # drop empty strings
    df["labels"] = df.cb_binary.astype(int)
    return df[["text", "labels"]].reset_index(drop=True)

def subsample_negatives(df, neg_per_pos, seed=SEED):
    pos = df[df.labels == 1]
    neg = df[df.labels == 0]
    keep = min(len(neg), max(len(pos) * neg_per_pos, 1))
    neg = neg.sample(keep, random_state=seed)
    return pd.concat([pos, neg]).sample(frac=1, random_state=seed).reset_index(drop=True)

PUB = PREP_DIR / "stage1_public.parquet"
DOM = PREP_DIR / "withu_domain.parquet"

s1_train = load_split(PUB, "stage1_train")
s1_val   = load_split(PUB, "stage1_val")
s1_test  = load_split(PUB, "stage1_test")

d_train_full = load_split(DOM, "domain_train")
d_train = subsample_negatives(d_train_full, S2_NEG_PER_POS)   # balance-ish for stage 2
d_val   = load_split(DOM, "domain_val")
d_test  = load_split(DOM, "domain_test")

if SMOKE_TEST:
    s1_train = s1_train.sample(1500, random_state=SEED)
    s1_val   = s1_val.sample(400, random_state=SEED)
    d_train  = d_train.sample(min(len(d_train), 800), random_state=SEED)

for name, df in [("s1_train", s1_train), ("s1_val", s1_val), ("s1_test", s1_test),
                 ("d_train(bal)", d_train), ("d_val", d_val), ("d_test", d_test)]:
    print(f"{name:14s} n={len(df):6d}  pos={int(df.labels.sum()):5d}  pos_rate={df.labels.mean():.3f}")

s1_train       n= 93764  pos=43224  pos_rate=0.461
s1_val         n= 11721  pos= 5404  pos_rate=0.461
s1_test        n= 11721  pos= 5403  pos_rate=0.461
d_train(bal)   n=  4203  pos=  467  pos_rate=0.111
d_val          n=  7636  pos=   38  pos_rate=0.005
d_test         n=  7631  pos=   32  pos_rate=0.004


## 2. Tokenizer + class weights (imbalance handling)

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_ds(df):
    ds = Dataset.from_pandas(df, preserve_index=False)
    ds = ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN),
                batched=True)
    return ds

from sklearn.utils.class_weight import compute_class_weight
def class_weights(df):
    cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=df.labels.values)
    return torch.tensor(cw, dtype=torch.float)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    p, r, f, _ = precision_recall_fscore_support(labels, preds, average="binary",
                                                 pos_label=1, zero_division=0)
    return {"accuracy": accuracy_score(labels, preds),
            "precision": p, "recall": r, "f1": f}

class WeightedTrainer(Trainer):
    """CrossEntropy with class weights to handle toxic/clean imbalance."""
    def __init__(self, class_weights=None, **kw):
        super().__init__(**kw)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(outputs.logits.device))
        loss = loss_fct(outputs.logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [8]:
def make_args(output_dir, epochs, bs, lr):
    """TrainingArguments with a shim for the eval_strategy/evaluation_strategy rename."""
    common = dict(output_dir=str(output_dir), num_train_epochs=epochs,
                  per_device_train_batch_size=bs, per_device_eval_batch_size=64,
                  learning_rate=lr, weight_decay=0.01, warmup_ratio=0.1,
                  logging_steps=50, save_total_limit=1, seed=SEED,
                  load_best_model_at_end=True, metric_for_best_model="f1",
                  report_to="none")
    try:
        return TrainingArguments(eval_strategy="epoch", save_strategy="epoch", **common)
    except TypeError:                               # older transformers
        return TrainingArguments(evaluation_strategy="epoch", save_strategy="epoch", **common)

## 3. Stage-1 — general toxicity on the public pool

Fine-tune KcELECTRA (fresh binary head) on BEEP + K-MHaS. Early-stops on the public val F1 and saves
the best checkpoint to `./models/stage1_general/`.


In [15]:
s1_tr_ds, s1_va_ds, s1_te_ds = to_ds(s1_train), to_ds(s1_val), to_ds(s1_test)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

s1_trainer = WeightedTrainer(
    class_weights=class_weights(s1_train),
    model=model,
    args=make_args(STAGE1_DIR, S1_EPOCHS if not SMOKE_TEST else 1, S1_BS, S1_LR),
    train_dataset=s1_tr_ds, eval_dataset=s1_va_ds,
    processing_class=tokenizer, compute_metrics=compute_metrics,
)
s1_trainer.train()
print("Stage-1 public TEST:", s1_trainer.evaluate(s1_te_ds))
s1_trainer.save_model(str(STAGE1_DIR)); tokenizer.save_pretrained(str(STAGE1_DIR))
print("saved ->", STAGE1_DIR)

Map:   0%|          | 0/93764 [00:00<?, ? examples/s]

Map:   0%|          | 0/11721 [00:00<?, ? examples/s]

Map:   0%|          | 0/11721 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: beomi/KcELECTRA-base
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.weight                           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[tran

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

## 4. Stage-2 — domain adaptation on WithU

Reload the Stage-1 checkpoint and continue training on the (negative-subsampled) domain data at a
**lower learning rate**. This nudges the model toward kids' group-chat language without erasing what
Stage-1 learned. Best checkpoint (by domain-val F1) is saved to `./models/stage2_domain_final/`.


In [12]:
d_tr_ds, d_va_ds, d_te_ds = to_ds(d_train), to_ds(d_val), to_ds(d_test)

model2 = AutoModelForSequenceClassification.from_pretrained(str(STAGE1_DIR), num_labels=2)

s2_trainer = WeightedTrainer(
    class_weights=class_weights(d_train),
    model=model2,
    args=make_args(STAGE2_DIR, S2_EPOCHS if not SMOKE_TEST else 1, S1_BS, S2_LR),
    train_dataset=d_tr_ds, eval_dataset=d_va_ds,
    processing_class=tokenizer, compute_metrics=compute_metrics,
)
s2_trainer.train()
s2_trainer.save_model(str(STAGE2_DIR)); tokenizer.save_pretrained(str(STAGE2_DIR))
print("saved final model ->", STAGE2_DIR)

Map:   0%|          | 0/4203 [00:00<?, ? examples/s]

Map:   0%|          | 0/7636 [00:00<?, ? examples/s]

Map:   0%|          | 0/7631 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.675108,0.525325,0.776585,0.009412,0.421053,0.018412
2,0.603551,0.530283,0.797407,0.010383,0.421053,0.020266


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved final model -> models/stage2_domain_final


## 5. Evaluation — the honest part

Report on **both** test sets:
- **Domain test** (your event-level hold-out): the number that matters, but it rests on a single WCB
  event with ~30 positives — treat as **directional**, quote it with that caveat.
- **Public test** (~11.7K rows): statistically powered check that the text head learned general toxicity.

We also sweep the decision threshold on **domain-val** (not test) and pick the F1-optimal operating
point for this head. (The .75/.85 gates in the design doc live at the *ensemble* stage; this module
just emits a calibrated P(toxic).)


In [13]:
def probs_and_labels(trainer, ds):
    out = trainer.predict(ds)
    logits = torch.tensor(out.predictions)
    p_tox = torch.softmax(logits, dim=-1)[:, 1].numpy()
    return p_tox, out.label_ids

def report(name, y, yhat):
    print(f"\n===== {name} =====")
    print(classification_report(y, yhat, target_names=["clean","toxic"], digits=4, zero_division=0))
    print("confusion [rows=true, cols=pred]:\n", confusion_matrix(y, yhat))

# threshold tuned on domain-val
pv, yv = probs_and_labels(s2_trainer, d_va_ds)
prec, rec, thr = precision_recall_curve(yv, pv)
f1s = 2*prec*rec/(prec+rec+1e-9)
best_t = float(thr[max(0, f1s[:-1].argmax())]) if len(thr) else 0.5
print(f"F1-optimal threshold on domain-val: {best_t:.3f}")

for nm, ds in [("DOMAIN test (directional)", d_te_ds), ("PUBLIC test (powered)", s1_te_ds)]:
    p, y = probs_and_labels(s2_trainer, ds)
    report(nm, y, (p >= best_t).astype(int))

metrics_out = {"threshold": best_t, "model": MODEL_NAME,
               "note": "domain test is single-event, directional only"}
(STAGE2_DIR / "eval_metrics.json").write_text(json.dumps(metrics_out, ensure_ascii=False, indent=2))
print("\nwrote", STAGE2_DIR / "eval_metrics.json")

F1-optimal threshold on domain-val: 0.844



===== DOMAIN test (directional) =====
              precision    recall  f1-score   support

       clean     0.9959    0.9862    0.9910      7599
       toxic     0.0094    0.0312    0.0145        32

    accuracy                         0.9822      7631
   macro avg     0.5027    0.5087    0.5028      7631
weighted avg     0.9917    0.9822    0.9869      7631

confusion [rows=true, cols=pred]:
 [[7494  105]
 [  31    1]]



===== PUBLIC test (powered) =====
              precision    recall  f1-score   support

       clean     0.8005    0.9747    0.8790      6318
       toxic     0.9603    0.7159    0.8203      5403

    accuracy                         0.8554     11721
   macro avg     0.8804    0.8453    0.8496     11721
weighted avg     0.8741    0.8554    0.8519     11721

confusion [rows=true, cols=pred]:
 [[6158  160]
 [1535 3868]]

wrote models/stage2_domain_final/eval_metrics.json


## 6. Inference helper

In [14]:
@torch.no_grad()
def predict_toxicity(texts, model_dir=STAGE2_DIR, threshold=None):
    tok = AutoTokenizer.from_pretrained(str(model_dir))
    mdl = AutoModelForSequenceClassification.from_pretrained(str(model_dir)).to(DEVICE).eval()
    if isinstance(texts, str): texts = [texts]
    enc = tok(texts, return_tensors="pt", truncation=True, max_length=MAX_LEN, padding=True).to(DEVICE)
    p = torch.softmax(mdl(**enc).logits, dim=-1)[:, 1].cpu().numpy()
    thr = threshold if threshold is not None else json.loads((Path(model_dir)/"eval_metrics.json").read_text())["threshold"]
    return [{"text": t, "p_toxic": float(x), "toxic": bool(x >= thr)} for t, x in zip(texts, p)]

# example
for r in predict_toxicity(["오늘 급식 뭐야?", "너 진짜 꺼져 아무도 너 안좋아해"]):
    print(r)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'text': '오늘 급식 뭐야?', 'p_toxic': 0.5318203568458557, 'toxic': False}
{'text': '너 진짜 꺼져 아무도 너 안좋아해', 'p_toxic': 0.8934651017189026, 'toxic': True}


## 7. What you have, and Phase 2+ next steps

**Saved:** `./models/stage2_domain_final/` — the domain-adapted text detector (`predict_toxicity`)
plus `eval_metrics.json` (operating threshold + caveat).

**This head is module A of the ensemble.** To complete the system from the design doc:
- **RAG slang layer** — wrap `predict_toxicity` with a researcher-maintained 은어/초성체/변형 비속어 lookup so new coinages are caught without retraining.
- **Phase 2 — context/sentiment module** (KLUE-RoBERTa over a 5–10 message window) using the AI-Hub 감정분석 + multi-turn sets.
- **Phase 4 — bystander model** (방어/동조/방관) bootstrapped as an LLM few-shot classifier from your labeling rules, with 방관 handled by log rules — this is where your WithU labels are the only real anchor.
- **Ensemble + API** — combine module scores → `cb_score`, apply the .75/.85 gates, serve behind FastAPI.

**Keep repeating the caveat in any writeup:** the powered evidence for this head is the public test
set; the domain test is one event and is directional until Phases 3–4 add collected/synthetic events.
